# 3. InferCNV 验证 Tumor vs Macrophage 注释

**目标**: 用 infercnvpy 推断 CNV，验证 clusters 5/7 是 Macrophage 还是 Tumor
- 肿瘤细胞应有明显 CNV (如 chr7 gain, chr10 loss — GBM 经典)  
- 免疫细胞应保持基本二倍体

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.set_figure_params(dpi=100, frameon=False)
%config InlineBackend.figure_format = 'retina'

FIG_DIR = Path('./figures/3_CNV_validation')
FIG_DIR.mkdir(parents=True, exist_ok=True)

### 1. 加载数据

In [ ]:
adata = sc.read_h5ad('./write/0_integrate_two_datasets/adata2.h5ad')
print(f'{adata.shape[0]} cells × {adata.shape[1]} genes')
print(adata.obs['cell_type'].value_counts())

### 2. 基因位置注释 (GENCODE v44 GTF)

In [ ]:
gtf_path = '../snGBM/results/gencode.v44.basic.annotation.gtf.gz'
cnv.io.genomic_position_from_gtf(gtf_path, adata=adata, gtf_gene_id='gene_name')

# 检查注释覆盖率
has_chr = adata.var['chromosome'].notna()
print(f'有染色体注释的基因: {has_chr.sum()} / {adata.n_vars} ({has_chr.mean():.1%})')
print(adata.var[has_chr][['chromosome', 'start', 'end']].head(10))

### 3. 运行 InferCNV

以 Endothelial + NK/T + Pericyte 作为正常参考 (最可靠的非恶性细胞群)

In [ ]:
# 正常参考细胞: 选血管 + 淋巴细胞 (最不可能被污染)
ref_types = ['Endothelial', 'NK/T', 'Pericyte']
n_ref = adata.obs['cell_type'].isin(ref_types).sum()
print(f'参考细胞数: {n_ref} ({ref_types})')

cnv.tl.infercnv(
    adata,
    reference_key='cell_type',
    reference_cat=ref_types,
    window_size=250,
    step=10,
    dynamic_threshold=1.5,
    n_jobs=4,
)
print('✅ InferCNV 完成')
print(f'CNV matrix shape: {adata.obsm["X_cnv"].shape}')

### 4. CNV Score 计算

In [ ]:
# CNV 空间: PCA → neighbors → Leiden → score
cnv.tl.pca(adata)
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata, resolution=0.5)

cnv.tl.cnv_score(adata)
print(f'CNV score range: {adata.obs["cnv_score"].min():.4f} - {adata.obs["cnv_score"].max():.4f}')
print('\n按 cell_type 的 CNV score:')
print(adata.obs.groupby('cell_type')['cnv_score'].agg(['mean', 'median', 'std']).sort_values('mean', ascending=False).round(4))

### 5. CNV Score 可视化

In [ ]:
# CNV score on UMAP
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cnv_score', ax=axes[0], frameon=False,
           title='CNV Score', cmap='RdBu_r', vcenter=0, show=False)
sc.pl.umap(adata, color='cell_type', ax=axes[1], frameon=False,
           title='Cell Type', legend_loc='on data', legend_fontsize=7, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_cnv_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 按 cell_type 的 CNV score 箱线图
order = adata.obs.groupby('cell_type')['cnv_score'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=adata.obs, x='cell_type', y='cnv_score', order=order, ax=ax, fliersize=0.5)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('CNV Score by Cell Type')
plt.tight_layout()
plt.savefig(FIG_DIR / 'boxplot_cnv_score_by_celltype.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 按 leiden cluster 的 CNV score, 标注 cell_type
df_plot = adata.obs[['leiden', 'cell_type', 'cnv_score']].copy()
df_plot['label'] = df_plot['leiden'].astype(str) + ' (' + df_plot['cell_type'].astype(str) + ')'
order_cl = df_plot.groupby('label')['cnv_score'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(16, 5))
sns.boxplot(data=df_plot, x='label', y='cnv_score', order=order_cl, ax=ax, fliersize=0.3)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha='right', fontsize=8)
ax.set_title('CNV Score by Leiden Cluster (annotated cell type)')

# 高亮 clusters 5 和 7
for i, label in enumerate(order_cl):
    if label.startswith('5 ') or label.startswith('7 '):
        ax.get_xticklabels()[i].set_color('red')
        ax.get_xticklabels()[i].set_fontweight('bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'boxplot_cnv_score_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. 染色体级 CNV Heatmap

In [ ]:
# 按 cell_type 汇总的染色体 CNV heatmap
cnv.pl.chromosome_heatmap_summary(adata, groupby='cell_type', show=True, 
                                   save='_celltype.png')

In [ ]:
# 按 leiden 汇总的染色体 CNV heatmap
cnv.pl.chromosome_heatmap_summary(adata, groupby='leiden', show=True,
                                   save='_leiden.png')

### 7. 聚焦 Clusters 5 & 7 vs 已知 Tumor/Immune

In [ ]:
# 对比: 已知 tumor (8/9) vs 已知 immune (0/6/3) vs 待验证 (5/7)
focus_clusters = ['8', '9', '0', '6', '3', '5', '7', '2', '4']
adata_focus = adata[adata.obs['leiden'].isin(focus_clusters)].copy()

# 添加分组标签
group_map = {
    '8': 'Known_Tumor', '9': 'Known_Tumor',
    '0': 'Known_Immune', '6': 'Known_Immune', '3': 'Known_Immune',
    '2': 'Macrophage_assigned', '4': 'Macrophage_assigned',
    '5': 'Verify_cl5', '7': 'Verify_cl7',
}
adata_focus.obs['verify_group'] = adata_focus.obs['leiden'].map(group_map).astype('category')

fig, ax = plt.subplots(figsize=(10, 5))
order_v = ['Known_Tumor', 'Verify_cl5', 'Verify_cl7', 'Macrophage_assigned', 'Known_Immune']
sns.violinplot(data=adata_focus.obs, x='verify_group', y='cnv_score',
               order=order_v, ax=ax, inner='box', cut=0)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_title('CNV Score: Known Tumor vs Known Immune vs Clusters 5/7')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_cnv_verify.png', dpi=150, bbox_inches='tight')
plt.show()

# 统计检验
from scipy.stats import mannwhitneyu
for cl in ['5', '7']:
    cl_scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    tumor_scores = adata.obs.loc[adata.obs['leiden'].isin(['8', '9']), 'cnv_score']
    immune_scores = adata.obs.loc[adata.obs['leiden'].isin(['0', '3', '6']), 'cnv_score']
    _, p_tumor = mannwhitneyu(cl_scores, tumor_scores, alternative='two-sided')
    _, p_immune = mannwhitneyu(cl_scores, immune_scores, alternative='two-sided')
    print(f'Cluster {cl}: median={cl_scores.median():.4f}, '
          f'vs Tumor p={p_tumor:.2e}, vs Immune p={p_immune:.2e}')

### 8. 结论

In [ ]:
# 最终判定
tumor_median = adata.obs.loc[adata.obs['cell_type'] == 'Tumor', 'cnv_score'].median()
immune_median = adata.obs.loc[adata.obs['cell_type'].isin(['Endothelial', 'NK/T', 'Pericyte']), 'cnv_score'].median()
threshold = (tumor_median + immune_median) / 2

for cl in ['5', '7', '2', '4']:
    med = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score'].median()
    verdict = 'TUMOR-like' if med > threshold else 'IMMUNE-like'
    print(f'Cluster {cl} ({adata.obs.loc[adata.obs["leiden"]==cl, "cell_type"].iloc[0]}): '
          f'median CNV={med:.4f} → {verdict}')

print(f'\n参考: Tumor median={tumor_median:.4f}, Immune median={immune_median:.4f}, threshold={threshold:.4f}')

### 延伸：检查 Neuronal clusters 15/18 的 CNV 信号

染色体热图显示 cluster 15 和 18 有明显 CNV 信号（尤其 chr7 区域），需确认是否混入肿瘤细胞。

In [ ]:
# Cluster 15/18 基本信息
for cl in ['15', '18']:
    mask = adata.obs['leiden'] == cl
    ct = adata.obs.loc[mask, 'cell_type'].value_counts()
    n = mask.sum()
    med = adata.obs.loc[mask, 'cnv_score'].median()
    print(f"=== Cluster {cl} (n={n}, median CNV={med:.4f}) ===")
    print(ct)
    print()

In [ ]:
# Neuronal clusters vs Tumor vs Immune 的 CNV 分布对比
from scipy.stats import mannwhitneyu

suspect_cls = ['15', '18']
known_tumor_cls = ['8', '9', '17']
known_immune_cls = ['0', '3', '6']

tumor_scores = adata.obs.loc[adata.obs['leiden'].isin(known_tumor_cls), 'cnv_score']
immune_scores = adata.obs.loc[adata.obs['leiden'].isin(known_immune_cls), 'cnv_score']

# 统计检验
for cl in suspect_cls:
    cl_scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    _, p_vs_tumor = mannwhitneyu(cl_scores, tumor_scores, alternative='two-sided')
    _, p_vs_immune = mannwhitneyu(cl_scores, immune_scores, alternative='two-sided')
    print(f"Cluster {cl}: median={cl_scores.median():.4f}, "
          f"vs Tumor p={p_vs_tumor:.2e}, vs Immune p={p_vs_immune:.2e}")

# Violin plot
groups = {
    'Known_Tumor': known_tumor_cls,
    'Neuronal_cl15': ['15'],
    'Neuronal_cl18': ['18'],
    'Known_Immune': known_immune_cls,
}
records = []
for label, cls in groups.items():
    mask = adata.obs['leiden'].isin(cls)
    for s in adata.obs.loc[mask, 'cnv_score']:
        records.append({'group': label, 'cnv_score': s})
df_neuro = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df_neuro, x='group', y='cnv_score', ax=ax, inner='box', cut=0)
ax.axhline(y=0.0099, color='red', linestyle='--', alpha=0.6, label='threshold')
ax.set_title('CNV Score: Neuronal clusters 15/18 vs Tumor/Immune')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_neuronal_suspect.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 所有 cluster 的 CNV 汇总排名
summary = (adata.obs.groupby('leiden')['cnv_score']
           .agg(['median', 'mean', 'count'])
           .sort_values('median', ascending=False))
summary['cell_type'] = adata.obs.groupby('leiden')['cell_type'].agg(lambda x: x.value_counts().index[0])
summary['above_threshold'] = summary['median'] > 0.0099
print(summary.to_string())
print(f"\n阈值 = 0.0099 (Tumor median 与 Immune median 的中点)")

In [ ]:
# chr_pos 是 dict{chr_name: start_window_index}
chr_pos = adata.uns['cnv']['chr_pos']
n_windows = adata.obsm['X_cnv'].shape[1]

# 构建每条染色体的 (start, end) 区间
chrs_sorted = sorted(chr_pos.keys(), key=lambda x: int(x.replace('chr', '')))
chr_ranges = {}
for i, c in enumerate(chrs_sorted):
    start = chr_pos[c]
    end = chr_pos[chrs_sorted[i+1]] if i+1 < len(chrs_sorted) else n_windows
    chr_ranges[c] = (start, end)

print(f"chr7: windows [{chr_ranges['chr7'][0]}, {chr_ranges['chr7'][1]})")
print(f"chr10: windows [{chr_ranges['chr10'][0]}, {chr_ranges['chr10'][1]})")

# chr7 gain / chr10 loss per cluster
results = []
for cl in sorted(adata.obs['leiden'].unique(), key=int):
    mask = (adata.obs['leiden'] == cl).values
    X_cnv = adata.obsm['X_cnv'][mask]
    s7, e7 = chr_ranges['chr7']
    s10, e10 = chr_ranges['chr10']
    chr7_mean = X_cnv[:, s7:e7].mean()
    chr10_mean = X_cnv[:, s10:e10].mean()
    ct = adata.obs.loc[adata.obs['leiden'] == cl, 'cell_type'].mode()[0]
    results.append({
        'cluster': cl, 'cell_type': ct, 'n': mask.sum(),
        'chr7_gain': round(chr7_mean, 4),
        'chr10_loss': round(chr10_mean, 4),
    })

df_chr = pd.DataFrame(results).sort_values('chr7_gain', ascending=False)
print("\n--- Chr7 gain / Chr10 loss per cluster ---")
print(df_chr.to_string(index=False))

In [ ]:
# chr7 gain vs chr10 loss scatter plot (每个 cluster 一个点)
fig, ax = plt.subplots(figsize=(9, 7))

# 颜色映射
color_map = {
    'Tumor': 'red', 'Prol.tumor': 'salmon',
    'Neuronal': 'orange', 'Oligodendrocyte': 'purple',
    'Macrophage': 'steelblue', 'Microglia': 'dodgerblue',
    'Monocyte': 'skyblue', 'NK/T': 'cyan',
    'Endothelial': 'green', 'Pericyte': 'olive', 'Neutrophil': 'gray'
}

for _, row in df_chr.iterrows():
    c = color_map.get(row['cell_type'], 'black')
    ax.scatter(row['chr7_gain'], row['chr10_loss'],
               s=np.sqrt(row['n'])*3, c=c, edgecolors='k', linewidth=0.5, alpha=0.8)
    ax.annotate(f"cl{row['cluster']}", (row['chr7_gain'], row['chr10_loss']),
                fontsize=8, ha='left', va='bottom')

# 参考线
ax.axvline(x=0.008, color='red', linestyle='--', alpha=0.4, label='chr7 gain cutoff')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Mean chr7 CNV signal (gain →)')
ax.set_ylabel('Mean chr10 CNV signal (← loss)')
ax.set_title('Chr7 Gain vs Chr10 Loss per Cluster (bubble size = cell count)')

# Legend for cell types
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=v, markersize=8, label=k)
           for k, v in color_map.items() if k in df_chr['cell_type'].values]
ax.legend(handles=handles, loc='upper left', fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'scatter_chr7_chr10_per_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster 0 (Microglia) 的 CNV score 分布 — 是否双峰？
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cl, title in zip(axes, ['0', '15', '16'],
                          ['cl0 Microglia (n=12350)', 'cl15 Neuronal (n=1918)', 'cl16 Oligo (n=2844)']):
    scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    ax.hist(scores, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(x=0.0099, color='red', linestyle='--', alpha=0.6, label='threshold')
    ax.set_title(title)
    ax.set_xlabel('cnv_score')
    above = (scores > 0.0099).sum()
    ax.text(0.95, 0.95, f'{above}/{len(scores)} above thr\n({above/len(scores)*100:.1f}%)',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(FIG_DIR / 'hist_suspect_cnv_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### CNV 验证总结与注释修正建议

**已确认的注释：**
- ✅ Macrophage (cl2/4/5/7) — 正常二倍体，注释正确
- ✅ Tumor (cl8/9/17) — 强 CNV 信号，注释正确
- ✅ Prol.tumor (cl1) — 中等 CNV，注释正确
- ✅ 其他免疫/基质 (cl3/6/10/11/12/13/14) — 正常二倍体

**需要修正的注释：**
- ⚠️ **cl15 Neuronal (n=1918)**: 96.1% 超阈值 + chr7 gain → **应改为 NPC-like Tumor**
- ⚠️ **cl18 Neuronal (n=314)**: 同 cl15 → **应改为 NPC-like Tumor**
- ⚠️ **cl16 Oligodendrocyte (n=2844)**: 99.3% 超阈值 + chr7 gain → **应改为 OPC-like Tumor**
- ⚠️ **cl0 Microglia (n=12350)**: 34.3% 超阈值，双峰分布 → **需要 sub-clustering 分离真正 Microglia 和混入的肿瘤**

**生物学解释**：GBM 肿瘤细胞可表达四种转录程序（Neftel 2019）：MES-like、AC-like、NPC-like（类神经元）、OPC-like（类少突）。cl15/18 和 cl16 正是 NPC-like 和 OPC-like 程序的肿瘤细胞。

In [ ]:
# 注释修正方案（不修改 adata2，只生成建议列供后续使用）
cell_type_corrected = adata.obs['cell_type'].astype(str).copy()

# cl15/18 → NPC-like Tumor
cell_type_corrected.loc[(adata.obs['leiden'] == '15')] = 'NPC-like Tumor'
cell_type_corrected.loc[(adata.obs['leiden'] == '18')] = 'NPC-like Tumor'

# cl16 → OPC-like Tumor
cell_type_corrected.loc[(adata.obs['leiden'] == '16')] = 'OPC-like Tumor'

# cl0: 根据 cnv_score 阈值拆分
cl0_mask = adata.obs['leiden'] == '0'
cl0_tumor_mask = cl0_mask & (adata.obs['cnv_score'] > 0.0099)
cell_type_corrected.loc[cl0_tumor_mask] = 'Tumor (from cl0)'

adata.obs['cell_type_cnv_corrected'] = pd.Categorical(cell_type_corrected)

# 统计
print("=== 修正前后对比 ===")
print("\n原始注释:")
print(adata.obs['cell_type'].value_counts().to_string())
print("\nCNV 修正后:")
print(adata.obs['cell_type_cnv_corrected'].value_counts().to_string())

# 变化汇总
changed = (adata.obs['cell_type'].astype(str) != adata.obs['cell_type_cnv_corrected'].astype(str)).sum()
print(f"\n共修正 {changed} 个细胞 ({changed/len(adata)*100:.1f}%)")

In [ ]:
# UMAP: 原始 vs CNV 修正
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='原始注释', legend_loc='on data', legend_fontsize=7)
sc.pl.umap(adata, color='cell_type_cnv_corrected', ax=axes[1], show=False, title='CNV 修正后', legend_loc='on data', legend_fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_annotation_correction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 覆盖 cell_type 并保存
adata.obs['cell_type_original'] = adata.obs['cell_type'].copy()  # 备份
adata.obs['cell_type'] = adata.obs['cell_type_cnv_corrected'].copy()

# 保存 adata2.h5ad
adata.write_h5ad('adata2.h5ad')
print("已保存 adata2.h5ad")
print("\n修正后 cell_type:")
print(adata.obs['cell_type'].value_counts().to_string())

## 修正后全景可视化

In [ ]:
# 全景 UMAP — 修正后 cell_type
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# 左: 修正后
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False,
           title='Cell type (CNV-corrected)', frameon=False, legend_loc='right margin')
# 右: leiden 对照
sc.pl.umap(adata, color='leiden', ax=axes[1], show=False,
           title='Leiden clusters', frameon=False, legend_loc='on data',
           legend_fontsize=8)

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/umap_corrected_celltype.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 保存: umap_corrected_celltype.png")

In [ ]:
# Marker gene 定义 (与 notebook 2 一致)
gbm_markers = {
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN"],
    "Prol.tumor":     ["MKI67", "TOP2A", "CENPF", "PBK"],
    "NPC-like":       ["SYT1", "STMN2", "RBFOX3", "SLC17A7", "GAD1"],
    "OPC-like":       ["MAG", "CLDN11", "MOG", "APLP1"],
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD2"],
    "Pericyte":       ["COL1A2", "PDGFRB", "COL3A1"],
    "Endothelial":    ["ESAM", "FLT1", "VWF", "EGFL7"],
    "Monocyte":       ["FCN1", "VCAN", "S100A8", "S100A9"],
    "Microglia":      ["TMEM119", "CX3CR1", "P2RY12", "TREM2"],
    "Macrophage":     ["CD68", "LYZ", "C1QA", "C1QB", "APOE", "SPP1"],
    "Neutrophil":     ["FCGR3B", "CSF3R", "CXCR2"],
}

# 过滤缺失基因
markers_filt = {}
for ct, genes in gbm_markers.items():
    found = [g for g in genes if g in adata.var_names]
    markers_filt[ct] = found
    miss = set(genes) - set(found)
    if miss:
        print(f'⚠️ {ct}: 缺失 {miss}')

print(f'共 {sum(len(v) for v in markers_filt.values())} 个 marker 可用')

In [ ]:
# 重新计算 dendrogram（cell_type 从 11 变 12 类）
sc.tl.dendrogram(adata, groupby='cell_type')

# Dotplot: marker gene × 修正后 cell type
sc.pl.dotplot(adata, markers_filt, groupby='cell_type',
              standard_scale='var', dendrogram=True,
              save='_corrected_celltype_markers.png')
print("✅ 保存: dotplot_corrected_celltype_markers.png")

In [ ]:
# 细胞比例对比: 原始 vs 修正
import pandas as pd

df_orig = adata.obs['cell_type_original'].value_counts(normalize=True).rename('original')
df_corr = adata.obs['cell_type'].value_counts(normalize=True).rename('corrected')
df_comp = pd.concat([df_orig, df_corr], axis=1).fillna(0).sort_values('corrected', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左: 原始
df_orig_sorted = df_comp['original'].sort_values()
df_orig_sorted.plot.barh(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Original annotation')
axes[0].set_xlabel('Fraction')

# 右: 修正后
df_corr_sorted = df_comp['corrected'].sort_values()
df_corr_sorted.plot.barh(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('CNV-corrected annotation')
axes[1].set_xlabel('Fraction')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/barplot_celltype_proportion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 保存: barplot_celltype_proportion_comparison.png")

In [ ]:
### 合并 Tumor (from cl0) → Tumor

In [ ]:
# 合并 "Tumor (from cl0)" 到 "Tumor"
ct = adata.obs['cell_type'].astype(str)
ct[ct == 'Tumor (from cl0)'] = 'Tumor'
adata.obs['cell_type'] = pd.Categorical(ct)
print(adata.obs['cell_type'].value_counts())

In [ ]:
# 保存
adata.write_h5ad('adata2.h5ad')
print(f"✅ adata2.h5ad 已保存, {adata.n_obs} cells × {adata.n_vars} genes, {adata.obs['cell_type'].nunique()} cell types")

In [ ]:
# 合并后 UMAP
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Cell type (11 types)', frameon=False)
sc.pl.umap(adata, color='leiden', ax=axes[1], show=False, title='Leiden clusters', frameon=False, legend_loc='on data')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/umap_final_11types.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 保存: umap_final_11types.png")